In [ ]:
# %uv pip install pandas numpy matplotlib seaborn

In [ ]:
import glob

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1.データの読み込み

In [ ]:
# ぞれぞれのデータのカラムの順番が一致しているか確認

list_file_paths = sorted(glob.glob("data/unzipped/*.csv"))
list_cols = []

for path in list_file_paths:

    df = pd.read_csv(path)
    list_cols.append(df.columns.to_list())

print("カラム名・順番が一致しているかどうかの判定", all(lst == list_cols[0] for lst in list_cols))


In [ ]:
# まとめて読み込む
# それか、一個のファイルでデータの確認する・・・？

df_raw = pd.DataFrame()
list_dfs =[]
list_datecol_names = ["year", "month", "day", "hour"]

list_col_names = list_cols[0].copy() #カラム名と順番が一致しているので、最初の一個のリストを取り出す

for path in list_file_paths:

    cite_name = path.split("/")[2].split("_")[2]
    names = [i if i in list_datecol_names else f"{cite_name}_{i}" for i in list_col_names]

    # リストにまとめてconcatする場合
    # 処理とメモリ効率がいい。ただ、concatしたときに時間があってるかの保証がない。
    # df = pd.read_csv(path, names=names, skiprows=1)
    # list_dfs.append(df)

    # 年月日をkeyにouter joinする場合。（実質的に共通keyの時間を合わせておくことで後の分析が楽になる）
    df = pd.read_csv(path, names=names, skiprows=1)

    df.insert(0, "datetime", pd.to_datetime(df[list_datecol_names]))
    df = df.drop(list_datecol_names, axis=1) # datetimeに集約されている+mergeするときにカラムが増殖するから削除する

    if df_raw.empty:
        df_raw = df
    else:
        df_raw = pd.merge(df_raw, df, on="datetime", how="outer")

# リストにまとめてからconcatする場合
# df_raw = pd.concat(list_dfs, axis=0, ignore_index=True)

# それぞれの元ファイルごとの行番号を示すNoのカラムは削除する
# stationは観測地点の名称。ファイルごとに一意であり、カラム名にciteをprefixとしてつけているので、削除。
df_raw = df_raw.drop(columns=[c for c in df_raw.columns if "station" in c or "No" in c])


In [ ]:
df_raw.head(3)

In [ ]:
df_raw.shape

In [ ]:
df_raw.dtypes

## 2. データの中身の確認

In [ ]:
df_eda = df_raw.copy()

In [ ]:
# list_col_names_eda = list_col_names.copy()
list_col_names_eda = [i for i in list_col_names if i not in ["No", "year", "month", "day", "hour", "station"]]
# list_col_names_eda.append("datetime")

In [ ]:
list_col_names_eda

In [ ]:
# for col in list_col_names_eda:
#     print(df_eda.filter(like=col).describe())

In [ ]:
def draw_boxplot_and_check_null(df, col_name, unit):

    # グラフのスタイルとサイズを設定
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(14, 6))

    # _PM2.5 の文字を消して地点がわかりやすいようにするため、columnsをリネーム
    df_filtered = df.filter(like=col_name)
    df_filtered.columns = df_filtered.columns.str.replace(f"_{col_name}", "")

    sns.boxplot(data=df_filtered, palette="coolwarm")

    # 見た目の微調整
    plt.title(f"{col_name} Overview by Location", fontsize=16, pad=15)
    plt.ylabel(f"{col_name} {unit}", fontsize=12)
    plt.xlabel("Location", fontsize=12)
    plt.xticks(rotation=45)  # ラベルが被らないように斜めにする

    plt.tight_layout()
    plt.show()

    null_sum = df_filtered.isnull().sum()

    print(f"{col_name} nulls are:\n{null_sum}")

### 2.1 PM2.5

In [ ]:
df_eda.filter(like="PM2.5").describe()

In [ ]:
draw_boxplot_and_check_null(df=df_eda, col_name="PM2.5", unit="(μg/m³)")

# 上振れ方向に外れ値が多い
# ３桁が大半で、４桁はほぼない。桁がほとんど同じなので、対数変換までは不要。
# クリッピングやスケーリングなどの前処理は使用する予測モデルによって行うが、外れ値というよりも、大気の状況であるため、値が大きいことにも
# 一定の意味があるので安易はクリッピングはNG

### 2.2 PM10

In [ ]:
df_eda.filter(like="PM10").describe()

In [ ]:
draw_boxplot_and_check_null(df=df_eda, col_name="PM10", unit="(μg/m³)")

# 上振れ方向に外れ値が多い
# ３桁が大半で、４桁はほぼない。桁がほとんど同じなので、対数変換までは不要。
# クリッピングやスケーリングなどの前処理は使用する予測モデルによって行うが、外れ値というよりも、大気の状況であるため、値が大きいことにも
# 一定の意味があるので安易はクリッピングはNG

### 2.3 SO2

In [ ]:
df_eda.filter(like="SO2").describe()

In [ ]:
draw_boxplot_and_check_null(df=df_eda, col_name="SO2", unit="(μg/m³)")

# 上振れ方向に外れ値が多い
# ３桁が大半で、４桁はほぼない。桁がほとんど同じなので、対数変換までは不要。
# クリッピングやスケーリングなどの前処理は使用する予測モデルによって行うが、外れ値というよりも、大気の状況であるため、値が大きいことにも
# 一定の意味があるので安易はクリッピングはNG

### 2.4 NO2

In [ ]:
df_eda.filter(like="NO2").describe()

In [ ]:
draw_boxplot_and_check_null(df=df_eda, col_name="NO2", unit="(μg/m³)")

# 上振れ方向に外れ値が多い
# ほとんど~３００までの数値が占める。
# クリッピングやスケーリングなどの前処理は使用する予測モデルによって行うが、外れ値というよりも、大気の状況であるため、値が大きいことにも
# 一定の意味があるので安易はクリッピングはNG

### 2.5 CO

In [ ]:
df_eda.filter(like="CO").describe()

In [ ]:
draw_boxplot_and_check_null(df=df_eda, col_name="CO", unit="(μg/m³)")

# 2,3桁〜5桁まである。
# 桁が違うため、予測モデルによっては、対数変換が必要。

### 2.6 O3

In [ ]:
df_eda.filter(like="O3").describe()

In [ ]:
draw_boxplot_and_check_null(df=df_eda, col_name="O3", unit="(μg/m³)")

# 上振れした外れ値がおおい。
# 大半が3桁の値で、４桁の値は少数
# 外れ値込みでも、５００以下には大半がおさまる。一部１０００を超えているが、少数。
# モデルによってはクリッピングも候補となるが、本データでは大気汚染イベント自体が重要な情報となるため、まずは元データのまま扱う。

### 2.7 TEMP

In [ ]:
df_eda.head(3)

In [ ]:
df_eda.filter(like="TEMP").describe()

In [ ]:
draw_boxplot_and_check_null(df=df_eda, col_name="TEMP", unit="(degree Celsius)")

# 地点ごとに大きな差がない。

### 2.8 PRES

In [ ]:
df_eda.filter(like="PRES").describe()


In [ ]:
draw_boxplot_and_check_null(df=df_eda, col_name="PRES", unit="(hPa)")

# TEMP(気温)よりは、地点ごとに差がある

### 2.9 DEWP

In [ ]:
df_eda.filter(like="DEWP").describe()


In [ ]:
draw_boxplot_and_check_null(df=df_eda, col_name="DEWP", unit="(degree Celsius)")

### 2.10 RAIN

In [ ]:
df_eda.filter(like="RAIN").describe()

# 全ての地点で、少なくとも75%までは０がしめる

In [ ]:
df_eda.filter(like="RAIN").isnull().sum()

### 2.11 wd (wind direction)

In [ ]:
df_eda.filter(like="wd").describe()

# カテゴリ変数なので、学習の時にはカテゴリ型にするか、エンコードをする

In [ ]:
df_eda.filter(like="wd").isnull().sum()

### 2.12 WSPM (wind speed)

In [ ]:
df_eda.filter(like="WSPM").describe()


In [ ]:
draw_boxplot_and_check_null(df=df_eda, col_name="WSPM", unit="(m/s)")

# 風速単体よりも、風向きとセットで効果がある特徴量
# 元の情報量をなるべく減らさないように、風速単体での安易にクリッピングはしない

### 2.13 datetime

In [ ]:
df_eda["datetime"].max()

In [ ]:
df_eda["datetime"].min()


## 3. データの関係性の確認

In [ ]:
# 相関とかみてみる

# 相関係数の計算
corr_matrix = df_eda.corr(numeric_only=True)
threshold = 0.5  # 相関係数の絶対値がこの値以上のものを残す

# 自分自身の相関（常に1）を除外するため、対角成分を一時的に書き換える
corr_matrix_no_diag = corr_matrix.copy()
for i in range(len(corr_matrix_no_diag)):
    corr_matrix_no_diag.iloc[i, i] = 0

# しきい値を超えるペアが1つでもある列（変数）を特定
strong_features = corr_matrix_no_diag.columns[(corr_matrix_no_diag.abs() >= threshold).any()]

# 閾値を超えた変数だけで相関行列を再作成
filtered_corr = corr_matrix.loc[strong_features, strong_features]

# 4. ヒートマップの描画
plt.figure(figsize=(10, 8))  # 抽出された変数の数に合わせてサイズを調整
sns.heatmap(
    filtered_corr, 
    # annot=True,
    annot=False, 
    cmap="coolwarm", 
    vmin=-1, 
    vmax=1, 
    fmt=".2f",
    linewidths=0.5,     # セル間に白い境界線を入れて見やすく
    square=True         # セルをきれいな正方形にする
)
plt.title(f"Highly Correlated Features (Abs >= {threshold})")
plt.tight_layout()
plt.show()

In [ ]:
# 相関行列の下半分（重複と対角成分）をマスキングしてシリーズに変換
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
stacked_corr = upper_tri.stack()

# 絶対値が0.9以上のペアだけを抽出して表示
strong_pairs = stacked_corr[stacked_corr.abs() >= 0.9]
print("--- 相関が0.9以上の変数ペア一覧 ---")
print(strong_pairs.sort_values(ascending=False))

In [ ]:
# 相関が非常に強いペアがおおい。
# 例えば、
# Tiantan_DEWP        Dongsi_DEWP          1.000000
# となり、同じ化学成分だと地点が異なる場合でも、相関が１で完全に比例の関係が成り立つ場合もある
# 地点間で同じ気象要素は非常に高い相関を持つことが確認できた。
# Featuretoolsによる自動特徴量生成の効果を確認するため、今回は Aotizhongxin 観測地点の PM2.5 を目的変数として使用する。
# Featuretoolsで別地点の情報を組み合わせることで、目的地点PM2.5予測の性能向上が期待できる。

In [ ]:
# 相関係数1が妥当かどうかの確認
(df_eda["Dongsi_DEWP"] == df_eda["Tiantan_DEWP"]).all()

# 全てが一致しているわけではない

In [ ]:
df_eda[["Dongsi_DEWP", "Tiantan_DEWP"]].head(20)

# headの20件は完全一致

In [ ]:
(df_eda["Dongsi_DEWP"] == df_eda["Tiantan_DEWP"]).value_counts()
# 20件一致しない値がある

In [ ]:
# read_csv時に同じファイルを読み込んでしまっているか、ファイルパスの確認
print(list_file_paths)

In [ ]:
df_eda[["Dongsi_DEWP", "Tiantan_DEWP"]][df_eda["Dongsi_DEWP"] != df_eda["Tiantan_DEWP"]]

# 一致していない値の確認
# nanが該当した。=だとそもそもnan同士は等しくないと定義されているので、falseになるのは妥当。

In [ ]:
# "Dongsi_DEWP", "Tiantan_DEWP"はコピーしたのか、同じデータ

In [ ]:
# 他の相関係数1を確認
stacked_corr[stacked_corr.abs() == 1]

## 4. 前処理したデータを保存

In [ ]:
df_eda.head(3)

In [ ]:
df_eda.to_csv("./data/prepro_data.csv", index=False, encoding="utf-8")